# 04 — Snapshots and analysis in JavaScript

`syncParticles()` exposes positions, ids, and types as typed arrays — zero
copies out of wasm memory unless you ask for them. That makes post-processing
in JavaScript cheap. Here we compute the mean-squared displacement (MSD) of a
Lennard-Jones liquid by hand.

In [ ]:
// Load lammps.js (served by this site under ./lammps/). Run this cell first.
// The site root is derived from wherever this code runs: the kernel iframe
// inherits the page URL ({site}/lab/…), the worker kernel lives under
// {site}/extensions/….
const base = globalThis.document?.baseURI ?? location.href;
globalThis.SITE ??= base.replace(/(extensions|lab|notebooks|files|tree|repl|consoles|edit)\/.*$/, "");
globalThis.LammpsClient ??= (await import(new URL("lammps/client.js", globalThis.SITE))).LammpsClient;
"lammps.js loaded ✓"

In [ ]:
// A tiny console "plot": unicode sparkline of an array of numbers.
globalThis.spark = (xs) => {
  const min = Math.min(...xs), max = Math.max(...xs), glyphs = "▁▂▃▄▅▆▇█";
  return xs.map((v) => glyphs[Math.min(7, Math.floor(((v - min) / ((max - min) || 1)) * 8))]).join("");
};
"spark() defined ✓"

## Set up a liquid and remember where every atom started

`{ copy: true }` detaches the arrays from wasm memory — required when you keep
data across further `run` commands (the default arrays are *views* that the
next run overwrites). Atom order can change between snapshots, so we index by
atom id.

In [ ]:
globalThis.lammps = await LammpsClient.create({ print: (line) => console.log(line) });
lammps.start();
lammps.runScript(`
  units         lj
  timestep      0.005
  atom_style    atomic
  lattice       fcc 0.8442
  region        box block 0 4 0 4 0 4
  create_box    1 box
  create_atoms  1 box
  mass          1 1.0
  velocity      all create 1.5 87287
  pair_style    lj/cut 2.5
  pair_coeff    1 1 1.0 1.0 2.5
  fix           1 all nvt temp 1.5 1.5 0.5
  thermo        500
  run           1000
`);

const start = lammps.syncParticles({ copy: true });
globalThis.ref0 = new Map();
for (let i = 0; i < start.count; i++) {
  ref0.set(Number(start.ids[i]), start.positions.slice(3 * i, 3 * i + 3));
}
console.log("reference positions stored for", ref0.size, "atoms");

## Run in chunks and accumulate MSD(t)

Positions here are unwrapped (they keep going when an atom crosses the
periodic boundary), which is exactly what MSD needs.

In [ ]:
globalThis.msd = [];
for (let chunk = 0; chunk < 15; chunk++) {
  lammps.runCommand("run 200");
  const snap = lammps.syncParticles();
  let sum = 0;
  for (let i = 0; i < snap.count; i++) {
    const ref = ref0.get(Number(snap.ids[i]));
    const dx = snap.positions[3 * i] - ref[0];
    const dy = snap.positions[3 * i + 1] - ref[1];
    const dz = snap.positions[3 * i + 2] - ref[2];
    sum += dx * dx + dy * dy + dz * dz;
  }
  msd.push(sum / snap.count);
}
console.log("MSD(t):", spark(msd));
console.log("final MSD:", msd.at(-1).toFixed(3), "σ²");

## Diffusion coefficient

For 3D diffusion, MSD(t) ≈ 6·D·t at long times — a straight-line fit of the
tail gives D (in LJ units). This is the Einstein relation, computed entirely
in the browser.

In [ ]:
// Least-squares slope over the second half of the curve (t in LJ time units).
const dtChunk = 200 * 0.005;
const pts = msd.map((y, i) => [(i + 1) * dtChunk, y]).slice(Math.floor(msd.length / 2));
const n = pts.length;
const sx = pts.reduce((a, p) => a + p[0], 0), sy = pts.reduce((a, p) => a + p[1], 0);
const sxx = pts.reduce((a, p) => a + p[0] * p[0], 0), sxy = pts.reduce((a, p) => a + p[0] * p[1], 0);
const slope = (n * sxy - sx * sy) / (n * sxx - sx * sx);
console.log("D ≈", (slope / 6).toFixed(4), "σ²/τ");
lammps.dispose();